# siebel_id_test

Case-driven tests for incremental output maintenance on the real `shebang` tables.

Rules:
- each case has one markdown cell and one code cell
- each case calls `run_incremental_update(...)` from `siebel_id_update.ipynb`
- cases evolve the same `shebang.direct_bank`, `shebang.ggm_np`, and `shebang.output` tables sequentially
- rerun the initialization cell to restore the original baseline before replaying the sequence

In [ ]:
from pathlib import Path
import json
import os
import re
import sys
from pyspark.sql import functions as F
from pyspark.sql.types import StructType

ROOT_DIR = Path.cwd()
if not (ROOT_DIR / 'use_cases').exists() and (ROOT_DIR.parent / 'use_cases').exists():
    ROOT_DIR = ROOT_DIR.parent

# Align notebook Spark runtime with repository shell tooling.
java17_home = Path('/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home')
if java17_home.exists():
    os.environ['JAVA_HOME'] = str(java17_home)
    os.environ['PATH'] = f"{java17_home / 'bin'}:{os.environ.get('PATH', '')}"
os.environ['SPARK_CONF_DIR'] = str(ROOT_DIR / 'spark_conf')

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

# Load siebel_id_update code cells without nbformat dependency.
update_nb_path = ROOT_DIR / 'notebooks' / 'siebel_id_update.ipynb'
with open(update_nb_path, 'r', encoding='utf-8') as f:
    update_nb = json.load(f)

for cell in update_nb.get('cells', []):
    if cell.get('cell_type') == 'code':
        code = '\n'.join(cell.get('source', []))
        exec(code, globals())

if 'run_incremental_update' not in globals():
    raise RuntimeError('Failed to load run_incremental_update from siebel_id_update.ipynb')

from scripts.shared_spark import build_spark_session

if 'spark' not in globals():
    spark = build_spark_session('siebel-id-incremental-test-notebook')

checkpoint_dir = ROOT_DIR / '.local' / 'spark' / 'checkpoints' / 'siebel_id_test'
checkpoint_dir.mkdir(parents=True, exist_ok=True)
try:
    spark.sparkContext.setCheckpointDir(str(checkpoint_dir))
except Exception:
    pass

WORK_DB = 'shebang'
STATE_DB = 'shebang_state'
INPUT_DIR = ROOT_DIR / 'use_cases' / 'siebel_id' / 'input'
BASELINE_SNAPSHOTS = globals().get('BASELINE_SNAPSHOTS', {})


def table_exists(full_table_name: str) -> bool:
    return spark.catalog.tableExists(full_table_name)


def ensure_shebang_tables_exist() -> None:
    spark.sql(f'CREATE DATABASE IF NOT EXISTS {WORK_DB}')

    required_tables = [f'{WORK_DB}.direct_bank', f'{WORK_DB}.ggm_np', f'{WORK_DB}.output']
    if all(table_exists(full_table_name) for full_table_name in required_tables):
        return

    def to_table_name(path_obj: Path) -> str:
        return re.sub(r'[^a-z0-9_]', '_', path_obj.stem.lower())

    for csv_path in sorted(INPUT_DIR.glob('*.csv')):
        table_name = to_table_name(csv_path)
        full_table_name = f'{WORK_DB}.{table_name}'
        df = (
            spark.read.option('header', True)
            .option('sep', '\t')
            .option('inferSchema', True)
            .csv(str(csv_path))
        )
        df.write.mode('overwrite').format('parquet').saveAsTable(full_table_name)


def snapshot_table(full_table_name: str) -> dict:
    df = spark.table(full_table_name)
    return {
        'schema': df.schema.jsonValue(),
        'rows': [row.asDict(recursive=True) for row in df.collect()],
    }


def restore_snapshot(full_table_name: str, snapshot: dict) -> None:
    schema = StructType.fromJson(snapshot['schema'])
    df = spark.createDataFrame(snapshot['rows'], schema=schema)
    overwrite_table_with_staging_swap(spark, full_table_name, df)


def ensure_baseline_snapshots() -> None:
    ensure_shebang_tables_exist()
    if BASELINE_SNAPSHOTS:
        return

    for full_table_name in [
        f'{WORK_DB}.direct_bank',
        f'{WORK_DB}.ggm_np',
        f'{WORK_DB}.output',
    ]:
        BASELINE_SNAPSHOTS[full_table_name] = snapshot_table(full_table_name)


def reset_sequence_to_baseline() -> None:
    ensure_baseline_snapshots()

    restore_snapshot(f'{WORK_DB}.direct_bank', BASELINE_SNAPSHOTS[f'{WORK_DB}.direct_bank'])
    restore_snapshot(f'{WORK_DB}.ggm_np', BASELINE_SNAPSHOTS[f'{WORK_DB}.ggm_np'])
    restore_snapshot(f'{WORK_DB}.output', BASELINE_SNAPSHOTS[f'{WORK_DB}.output'])

    spark.sql(f'DROP TABLE IF EXISTS {STATE_DB}.edge_snapshot')
    spark.sql(f'DROP TABLE IF EXISTS {STATE_DB}.seed_snapshot')
    spark.sql(f'DROP TABLE IF EXISTS {STATE_DB}.seed_membership')
    spark.catalog.clearCache()


def pair_set(df):
    return {(str(r['REL_ID']), str(r['REL_ID_REGIE_KLANT'])) for r in df.select('REL_ID', 'REL_ID_REGIE_KLANT').collect()}


def run_once():
    return run_incremental_update(spark, source_db=WORK_DB, target_db=WORK_DB, state_db=STATE_DB, show_samples=False)


def assert_delta(case_name: str, prev_pairs: set, cur_pairs: set, expected_inserted: list, expected_removed: list) -> None:
    inserted = sorted(cur_pairs - prev_pairs)
    removed = sorted(prev_pairs - cur_pairs)
    exp_inserted = sorted(expected_inserted)
    exp_removed = sorted(expected_removed)

    if inserted != exp_inserted or removed != exp_removed:
        raise AssertionError(
            f"{case_name} FAIL\nExpected inserted={exp_inserted} removed={exp_removed}\nActual inserted={inserted} removed={removed}"
        )

    print(f"{case_name}: PASS")


def assert_output_has_no_duplicates(case_name: str) -> None:
    output_df = spark.table(f"{WORK_DB}.output")
    total_rows = output_df.count()
    distinct_rows = output_df.dropDuplicates(['REL_ID', 'REL_ID_REGIE_KLANT']).count()

    if total_rows != distinct_rows:
        raise AssertionError(
            f"{case_name} FAIL\nOutput contains duplicate rows: total_rows={total_rows} distinct_rows={distinct_rows}"
        )

    print(f"{case_name}: output has no duplicate rows")


def to_display_pandas(df):
    preview_df = df
    for field in preview_df.schema.fields:
        if field.dataType.typeName() in {'timestamp', 'date'}:
            preview_df = preview_df.withColumn(field.name, F.col(field.name).cast('string'))
    return preview_df.toPandas()


def show_dataframe(title: str, df, order_by: list[str] | None = None) -> None:
    print(title)
    preview_df = df.orderBy(*order_by) if order_by else df
    display(to_display_pandas(preview_df))


def show_case_tables(case_name: str) -> None:
    direct_df = spark.table(f"{WORK_DB}.direct_bank")
    ggm_df = spark.table(f"{WORK_DB}.ggm_np")
    output_df = spark.table(f"{WORK_DB}.output")

    print(f"{case_name} input/output preview")
    show_dataframe('direct_bank', direct_df, order_by=['rel_id', 'edl_valid_from_dts'])
    show_dataframe('ggm_np', ggm_df, order_by=['rel_id', 'edl_valid_from_dts'])
    show_dataframe('output', output_df, order_by=['REL_ID'])


def update_column_with_fallback(full_table: str, col_name: str, condition, new_value) -> None:
    # SQL UPDATE may fail for local Parquet tables; this fallback preserves behavior.
    def _transform(df):
        return df.withColumn(col_name, F.when(condition, new_value).otherwise(F.col(col_name)))

    mutate_table_with_overwrite_swap(spark, full_table, _transform)


print('Test harness initialized for shebang')

Test harness initialized for shebang


In [40]:
reset_sequence_to_baseline()
baseline_stats = run_once()
show_case_tables('Baseline after initialization')
print('Baseline stats:', baseline_stats)

Baseline after initialization input/output preview
direct_bank


,rel_id,edl_valid_from_dts,edl_valid_to_dts,np_sbl_id,drc_bnk_f
0,11410074,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-7JI-23579,Y
1,11410074,2020-03-05 10:51:06,2023-08-28 07:05:17,1-7JI-23579,N
2,11410074,2023-08-28 07:05:17,9999-12-31 00:00:00,1-7JI-23579,N
3,117786942,2019-11-08 18:15:22.600224,2024-04-09 11:00:16.23149,1-1B89RM5A,N
4,117786942,2024-04-09 11:00:16.23149,9999-12-31 00:00:00,1-1B89RM5A,N
5,118220032,2019-11-08 18:15:22.600224,9999-12-31 00:00:00,1-1N09820A,Y
6,118492640,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-1TWAZ5GR,N
7,118492640,2020-03-05 10:51:06,2024-04-09 11:00:15,1-1TWAZ5GR,Y
8,118492640,2024-04-09 11:00:15,9999-12-31 00:00:00,1-1TWAZ5GR,Y


ggm_np


,rel_id,edl_valid_from_dts,edl_valid_to_dts,ikb_no,del_f
0,11410074,2015-04-02 16:38:24,2016-01-25 18:14:36,1-7JI-23579,N
1,11410074,2016-01-25 18:14:36,2016-02-14 05:19:56.921842,1-7JI-23579,N
2,11410074,2016-02-14 05:19:56.921842,2016-03-27 08:45:36,1-7JI-23579,N
3,11410074,2016-03-27 08:45:36,2016-04-01 12:59:00,1-7JI-23579,N
4,11410074,2016-04-01 12:59:00,2016-09-24 06:35:53.613126,1-7JI-23579,N
...,...,...,...,...,...
107,118492640,2024-02-16 23:34:22,2024-04-24 16:33:50,1-1TWAZ5GR,N
108,118492640,2024-04-24 16:33:50,2024-11-28 05:48:49.831784,1-1TWAZ5GR,N
109,118492640,2024-11-28 05:48:49.831784,2025-02-19 11:10:02.333993,1-1TWAZ5GR,N
110,118492640,2025-02-19 11:10:02.333993,2025-10-01 07:27:12.034435,1-1TWAZ5GR,N


output


,REL_ID_REGIE_KLANT,REL_ID
0,118220032,105181982
1,118492640,11410074
2,118220032,115379534
3,118492640,117786942
4,118220032,118220032
5,118492640,118492640


Baseline stats: {'changed_edges': 14, 'added_seeds': 2, 'removed_seeds': 0, 'impacted_rel_ids': 6, 'output_rows': 6}


## Sequence Initialization

Run this cell before the cases to restore the original `shebang` baseline captured from the current database state and synchronize incremental state.

## Case 1: new direct_bank active seed

Scenario:
- insert a new `direct_bank` row where `drc_bnk_f='Y'` and `edl_valid_to_dts='9999-12-31'`

Expected delta/output:
- inserted: `('130000001', '130000001')`
- removed: none

In [41]:
prev_pairs = pair_set(spark.table(f"{WORK_DB}.output"))

spark.sql(
    f"""
    INSERT INTO {WORK_DB}.direct_bank (rel_id, edl_valid_from_dts, edl_valid_to_dts, np_sbl_id, drc_bnk_f)
    VALUES (130000001, to_timestamp('2026-05-18 09:00:00'), to_timestamp('9999-12-31 00:00:00'), '1-Z-NEW1', 'Y')
    """
)

_ = run_once()
cur_pairs = pair_set(spark.table(f"{WORK_DB}.output"))
show_case_tables('Case 1')
assert_delta('Case 1', prev_pairs, cur_pairs, [('130000001', '130000001')], [])
assert_output_has_no_duplicates('Case 1')

Case 1 input/output preview
direct_bank


,rel_id,edl_valid_from_dts,edl_valid_to_dts,np_sbl_id,drc_bnk_f
0,11410074,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-7JI-23579,Y
1,11410074,2020-03-05 10:51:06,2023-08-28 07:05:17,1-7JI-23579,N
2,11410074,2023-08-28 07:05:17,9999-12-31 00:00:00,1-7JI-23579,N
3,117786942,2019-11-08 18:15:22.600224,2024-04-09 11:00:16.23149,1-1B89RM5A,N
4,117786942,2024-04-09 11:00:16.23149,9999-12-31 00:00:00,1-1B89RM5A,N
5,118220032,2019-11-08 18:15:22.600224,9999-12-31 00:00:00,1-1N09820A,Y
6,118492640,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-1TWAZ5GR,N
7,118492640,2020-03-05 10:51:06,2024-04-09 11:00:15,1-1TWAZ5GR,Y
8,118492640,2024-04-09 11:00:15,9999-12-31 00:00:00,1-1TWAZ5GR,Y
9,130000001,2026-05-18 09:00:00,9999-12-31 00:00:00,1-Z-NEW1,Y


ggm_np


,rel_id,edl_valid_from_dts,edl_valid_to_dts,ikb_no,del_f
0,11410074,2015-04-02 16:38:24,2016-01-25 18:14:36,1-7JI-23579,N
1,11410074,2016-01-25 18:14:36,2016-02-14 05:19:56.921842,1-7JI-23579,N
2,11410074,2016-02-14 05:19:56.921842,2016-03-27 08:45:36,1-7JI-23579,N
3,11410074,2016-03-27 08:45:36,2016-04-01 12:59:00,1-7JI-23579,N
4,11410074,2016-04-01 12:59:00,2016-09-24 06:35:53.613126,1-7JI-23579,N
...,...,...,...,...,...
107,118492640,2024-02-16 23:34:22,2024-04-24 16:33:50,1-1TWAZ5GR,N
108,118492640,2024-04-24 16:33:50,2024-11-28 05:48:49.831784,1-1TWAZ5GR,N
109,118492640,2024-11-28 05:48:49.831784,2025-02-19 11:10:02.333993,1-1TWAZ5GR,N
110,118492640,2025-02-19 11:10:02.333993,2025-10-01 07:27:12.034435,1-1TWAZ5GR,N


output


,REL_ID_REGIE_KLANT,REL_ID
0,118220032,105181982
1,118492640,11410074
2,118220032,115379534
3,118492640,117786942
4,118220032,118220032
5,118492640,118492640
6,130000001,130000001


Case 1: PASS
Case 1: output has no duplicate rows


## Case 2: flip seed off with fallback mutation

Scenario:
- change `drc_bnk_f` from `Y` to `N` for seed `rel_id=118492640`
- local SQL UPDATE may be unsupported, so fallback uses DataFrame overwrite staging swap

Expected delta/output:
- inserted: none
- removed: `('11410074', '118492640')`, `('117786942', '118492640')`, `('118492640', '118492640')`

In [42]:
prev_pairs = pair_set(spark.table(f"{WORK_DB}.output"))

cond = (
    (F.col('rel_id') == F.lit(118492640))
    & (F.col('np_sbl_id') == F.lit('1-1TWAZ5GR'))
    & (F.col('drc_bnk_f') == F.lit('Y'))
    & (F.col('edl_valid_to_dts') == F.to_timestamp(F.lit('9999-12-31 00:00:00')))
)
update_column_with_fallback(f"{WORK_DB}.direct_bank", 'drc_bnk_f', cond, F.lit('N'))

_ = run_once()
cur_pairs = pair_set(spark.table(f"{WORK_DB}.output"))
show_case_tables('Case 2')
assert_delta(
    'Case 2',
    prev_pairs,
    cur_pairs,
    [],
    [('11410074', '118492640'), ('117786942', '118492640'), ('118492640', '118492640')],
)
assert_output_has_no_duplicates('Case 2')

Case 2 input/output preview
direct_bank


,rel_id,edl_valid_from_dts,edl_valid_to_dts,np_sbl_id,drc_bnk_f
0,11410074,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-7JI-23579,Y
1,11410074,2020-03-05 10:51:06,2023-08-28 07:05:17,1-7JI-23579,N
2,11410074,2023-08-28 07:05:17,9999-12-31 00:00:00,1-7JI-23579,N
3,117786942,2019-11-08 18:15:22.600224,2024-04-09 11:00:16.23149,1-1B89RM5A,N
4,117786942,2024-04-09 11:00:16.23149,9999-12-31 00:00:00,1-1B89RM5A,N
5,118220032,2019-11-08 18:15:22.600224,9999-12-31 00:00:00,1-1N09820A,Y
6,118492640,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-1TWAZ5GR,N
7,118492640,2020-03-05 10:51:06,2024-04-09 11:00:15,1-1TWAZ5GR,Y
8,118492640,2024-04-09 11:00:15,9999-12-31 00:00:00,1-1TWAZ5GR,N
9,130000001,2026-05-18 09:00:00,9999-12-31 00:00:00,1-Z-NEW1,Y


ggm_np


,rel_id,edl_valid_from_dts,edl_valid_to_dts,ikb_no,del_f
0,11410074,2015-04-02 16:38:24,2016-01-25 18:14:36,1-7JI-23579,N
1,11410074,2016-01-25 18:14:36,2016-02-14 05:19:56.921842,1-7JI-23579,N
2,11410074,2016-02-14 05:19:56.921842,2016-03-27 08:45:36,1-7JI-23579,N
3,11410074,2016-03-27 08:45:36,2016-04-01 12:59:00,1-7JI-23579,N
4,11410074,2016-04-01 12:59:00,2016-09-24 06:35:53.613126,1-7JI-23579,N
...,...,...,...,...,...
107,118492640,2024-02-16 23:34:22,2024-04-24 16:33:50,1-1TWAZ5GR,N
108,118492640,2024-04-24 16:33:50,2024-11-28 05:48:49.831784,1-1TWAZ5GR,N
109,118492640,2024-11-28 05:48:49.831784,2025-02-19 11:10:02.333993,1-1TWAZ5GR,N
110,118492640,2025-02-19 11:10:02.333993,2025-10-01 07:27:12.034435,1-1TWAZ5GR,N


output


,REL_ID_REGIE_KLANT,REL_ID
0,118220032,105181982
1,118220032,115379534
2,118220032,118220032
3,130000001,130000001


Case 2: PASS
Case 2: output has no duplicate rows


## Case 3: ggm_np rel_id reassignment via fallback

Scenario:
- reassign `ggm_np.rel_id` from `130000002` to `130000003` for `ikb_no='1-1N09820A'`

Expected delta/output:
- inserted: `('130000003', '118220032')`
- removed: `('130000002', '118220032')`

In [43]:
spark.sql(
    f"""
    INSERT INTO {WORK_DB}.ggm_np (rel_id, edl_valid_from_dts, edl_valid_to_dts, ikb_no, del_f)
    VALUES (130000002, to_timestamp('2026-05-18 11:00:00'), to_timestamp('9999-12-31 00:00:00'), '1-1N09820A', 'N')
    """
)
_ = run_once()  # absorb inserted row first
prev_pairs = pair_set(spark.table(f"{WORK_DB}.output"))

cond = (
    (F.col('rel_id') == F.lit(130000002))
    & (F.col('ikb_no') == F.lit('1-1N09820A'))
    & (F.col('edl_valid_from_dts') == F.to_timestamp(F.lit('2026-05-18 11:00:00')))
)
update_column_with_fallback(f"{WORK_DB}.ggm_np", 'rel_id', cond, F.lit(130000003))

_ = run_once()
cur_pairs = pair_set(spark.table(f"{WORK_DB}.output"))
show_case_tables('Case 3')
assert_delta('Case 3', prev_pairs, cur_pairs, [('130000003', '118220032')], [('130000002', '118220032')])
assert_output_has_no_duplicates('Case 3')

Case 3 input/output preview
direct_bank


,rel_id,edl_valid_from_dts,edl_valid_to_dts,np_sbl_id,drc_bnk_f
0,11410074,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-7JI-23579,Y
1,11410074,2020-03-05 10:51:06,2023-08-28 07:05:17,1-7JI-23579,N
2,11410074,2023-08-28 07:05:17,9999-12-31 00:00:00,1-7JI-23579,N
3,117786942,2019-11-08 18:15:22.600224,2024-04-09 11:00:16.23149,1-1B89RM5A,N
4,117786942,2024-04-09 11:00:16.23149,9999-12-31 00:00:00,1-1B89RM5A,N
5,118220032,2019-11-08 18:15:22.600224,9999-12-31 00:00:00,1-1N09820A,Y
6,118492640,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-1TWAZ5GR,N
7,118492640,2020-03-05 10:51:06,2024-04-09 11:00:15,1-1TWAZ5GR,Y
8,118492640,2024-04-09 11:00:15,9999-12-31 00:00:00,1-1TWAZ5GR,N
9,130000001,2026-05-18 09:00:00,9999-12-31 00:00:00,1-Z-NEW1,Y


ggm_np


,rel_id,edl_valid_from_dts,edl_valid_to_dts,ikb_no,del_f
0,11410074,2015-04-02 16:38:24,2016-01-25 18:14:36,1-7JI-23579,N
1,11410074,2016-01-25 18:14:36,2016-02-14 05:19:56.921842,1-7JI-23579,N
2,11410074,2016-02-14 05:19:56.921842,2016-03-27 08:45:36,1-7JI-23579,N
3,11410074,2016-03-27 08:45:36,2016-04-01 12:59:00,1-7JI-23579,N
4,11410074,2016-04-01 12:59:00,2016-09-24 06:35:53.613126,1-7JI-23579,N
...,...,...,...,...,...
108,118492640,2024-04-24 16:33:50,2024-11-28 05:48:49.831784,1-1TWAZ5GR,N
109,118492640,2024-11-28 05:48:49.831784,2025-02-19 11:10:02.333993,1-1TWAZ5GR,N
110,118492640,2025-02-19 11:10:02.333993,2025-10-01 07:27:12.034435,1-1TWAZ5GR,N
111,118492640,2025-10-01 07:27:12.034435,9999-12-31 00:00:00,1-1TWAZ5GR,N


output


,REL_ID_REGIE_KLANT,REL_ID
0,118220032,105181982
1,118220032,115379534
2,118220032,118220032
3,130000001,130000001
4,118220032,130000003


Case 3: PASS
Case 3: output has no duplicate rows


## Case 4: idempotent no-change run

Scenario:
- execute incremental update twice with no source mutations

Expected delta/output:
- inserted: none
- removed: none

In [44]:
prev_pairs = pair_set(spark.table(f"{WORK_DB}.output"))

_ = run_once()  # no mutation
cur_pairs = pair_set(spark.table(f"{WORK_DB}.output"))
show_case_tables('Case 4')
assert_delta('Case 4', prev_pairs, cur_pairs, [], [])
assert_output_has_no_duplicates('Case 4')

Case 4 input/output preview
direct_bank


,rel_id,edl_valid_from_dts,edl_valid_to_dts,np_sbl_id,drc_bnk_f
0,11410074,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-7JI-23579,Y
1,11410074,2020-03-05 10:51:06,2023-08-28 07:05:17,1-7JI-23579,N
2,11410074,2023-08-28 07:05:17,9999-12-31 00:00:00,1-7JI-23579,N
3,117786942,2019-11-08 18:15:22.600224,2024-04-09 11:00:16.23149,1-1B89RM5A,N
4,117786942,2024-04-09 11:00:16.23149,9999-12-31 00:00:00,1-1B89RM5A,N
5,118220032,2019-11-08 18:15:22.600224,9999-12-31 00:00:00,1-1N09820A,Y
6,118492640,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-1TWAZ5GR,N
7,118492640,2020-03-05 10:51:06,2024-04-09 11:00:15,1-1TWAZ5GR,Y
8,118492640,2024-04-09 11:00:15,9999-12-31 00:00:00,1-1TWAZ5GR,N
9,130000001,2026-05-18 09:00:00,9999-12-31 00:00:00,1-Z-NEW1,Y


ggm_np


,rel_id,edl_valid_from_dts,edl_valid_to_dts,ikb_no,del_f
0,11410074,2015-04-02 16:38:24,2016-01-25 18:14:36,1-7JI-23579,N
1,11410074,2016-01-25 18:14:36,2016-02-14 05:19:56.921842,1-7JI-23579,N
2,11410074,2016-02-14 05:19:56.921842,2016-03-27 08:45:36,1-7JI-23579,N
3,11410074,2016-03-27 08:45:36,2016-04-01 12:59:00,1-7JI-23579,N
4,11410074,2016-04-01 12:59:00,2016-09-24 06:35:53.613126,1-7JI-23579,N
...,...,...,...,...,...
108,118492640,2024-04-24 16:33:50,2024-11-28 05:48:49.831784,1-1TWAZ5GR,N
109,118492640,2024-11-28 05:48:49.831784,2025-02-19 11:10:02.333993,1-1TWAZ5GR,N
110,118492640,2025-02-19 11:10:02.333993,2025-10-01 07:27:12.034435,1-1TWAZ5GR,N
111,118492640,2025-10-01 07:27:12.034435,9999-12-31 00:00:00,1-1TWAZ5GR,N


output


,REL_ID_REGIE_KLANT,REL_ID
0,118220032,105181982
1,118220032,115379534
2,118220032,118220032
3,130000001,130000001
4,118220032,130000003


Case 4: PASS
Case 4: output has no duplicate rows


## Case 5: new ikb_no for the same rel_id

Scenario:
- insert a new `ggm_np` row with a fresh `ikb_no` for existing `rel_id=118220032`

Expected delta/output:
- inserted: none
- removed: none
- the output row for `118220032` remains unique and unchanged

In [45]:
prev_pairs = pair_set(spark.table(f"{WORK_DB}.output"))

spark.sql(
    f"""
    INSERT INTO {WORK_DB}.ggm_np (rel_id, edl_valid_from_dts, edl_valid_to_dts, ikb_no, del_f)
    VALUES (118220032, to_timestamp('2026-05-18 12:00:00'), to_timestamp('9999-12-31 00:00:00'), '1-ALT-IKB-118220032', 'N')
    """
)

_ = run_once()
cur_pairs = pair_set(spark.table(f"{WORK_DB}.output"))
show_case_tables('Case 5')
assert_delta('Case 5', prev_pairs, cur_pairs, [], [])
assert_output_has_no_duplicates('Case 5')

Case 5 input/output preview
direct_bank


,rel_id,edl_valid_from_dts,edl_valid_to_dts,np_sbl_id,drc_bnk_f
0,11410074,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-7JI-23579,Y
1,11410074,2020-03-05 10:51:06,2023-08-28 07:05:17,1-7JI-23579,N
2,11410074,2023-08-28 07:05:17,9999-12-31 00:00:00,1-7JI-23579,N
3,117786942,2019-11-08 18:15:22.600224,2024-04-09 11:00:16.23149,1-1B89RM5A,N
4,117786942,2024-04-09 11:00:16.23149,9999-12-31 00:00:00,1-1B89RM5A,N
5,118220032,2019-11-08 18:15:22.600224,9999-12-31 00:00:00,1-1N09820A,Y
6,118492640,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-1TWAZ5GR,N
7,118492640,2020-03-05 10:51:06,2024-04-09 11:00:15,1-1TWAZ5GR,Y
8,118492640,2024-04-09 11:00:15,9999-12-31 00:00:00,1-1TWAZ5GR,N
9,130000001,2026-05-18 09:00:00,9999-12-31 00:00:00,1-Z-NEW1,Y


ggm_np


,rel_id,edl_valid_from_dts,edl_valid_to_dts,ikb_no,del_f
0,11410074,2015-04-02 16:38:24,2016-01-25 18:14:36,1-7JI-23579,N
1,11410074,2016-01-25 18:14:36,2016-02-14 05:19:56.921842,1-7JI-23579,N
2,11410074,2016-02-14 05:19:56.921842,2016-03-27 08:45:36,1-7JI-23579,N
3,11410074,2016-03-27 08:45:36,2016-04-01 12:59:00,1-7JI-23579,N
4,11410074,2016-04-01 12:59:00,2016-09-24 06:35:53.613126,1-7JI-23579,N
...,...,...,...,...,...
109,118492640,2024-04-24 16:33:50,2024-11-28 05:48:49.831784,1-1TWAZ5GR,N
110,118492640,2024-11-28 05:48:49.831784,2025-02-19 11:10:02.333993,1-1TWAZ5GR,N
111,118492640,2025-02-19 11:10:02.333993,2025-10-01 07:27:12.034435,1-1TWAZ5GR,N
112,118492640,2025-10-01 07:27:12.034435,9999-12-31 00:00:00,1-1TWAZ5GR,N


output


,REL_ID_REGIE_KLANT,REL_ID
0,118220032,105181982
1,118220032,115379534
2,118220032,118220032
3,130000001,130000001
4,118220032,130000003


Case 5: PASS
Case 5: output has no duplicate rows


## Case 6: duplicate source row does not duplicate output

Scenario:
- duplicate an existing active `ggm_np` row in the input

Expected delta/output:
- inserted: none
- removed: none
- output remains unique by `(REL_ID, REL_ID_REGIE_KLANT)`

In [46]:
prev_pairs = pair_set(spark.table(f"{WORK_DB}.output"))

spark.sql(
    f"""
    INSERT INTO {WORK_DB}.ggm_np
    SELECT rel_id, edl_valid_from_dts, edl_valid_to_dts, ikb_no, del_f
    FROM {WORK_DB}.ggm_np
    WHERE rel_id = 118220032
      AND ikb_no = '1-1N09820A'
      AND edl_valid_to_dts = to_timestamp('9999-12-31 00:00:00')
    LIMIT 1
    """
)

_ = run_once()
cur_pairs = pair_set(spark.table(f"{WORK_DB}.output"))
show_case_tables('Case 6')
assert_delta('Case 6', prev_pairs, cur_pairs, [], [])
assert_output_has_no_duplicates('Case 6')

Case 6 input/output preview
direct_bank


,rel_id,edl_valid_from_dts,edl_valid_to_dts,np_sbl_id,drc_bnk_f
0,11410074,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-7JI-23579,Y
1,11410074,2020-03-05 10:51:06,2023-08-28 07:05:17,1-7JI-23579,N
2,11410074,2023-08-28 07:05:17,9999-12-31 00:00:00,1-7JI-23579,N
3,117786942,2019-11-08 18:15:22.600224,2024-04-09 11:00:16.23149,1-1B89RM5A,N
4,117786942,2024-04-09 11:00:16.23149,9999-12-31 00:00:00,1-1B89RM5A,N
5,118220032,2019-11-08 18:15:22.600224,9999-12-31 00:00:00,1-1N09820A,Y
6,118492640,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-1TWAZ5GR,N
7,118492640,2020-03-05 10:51:06,2024-04-09 11:00:15,1-1TWAZ5GR,Y
8,118492640,2024-04-09 11:00:15,9999-12-31 00:00:00,1-1TWAZ5GR,N
9,130000001,2026-05-18 09:00:00,9999-12-31 00:00:00,1-Z-NEW1,Y


ggm_np


,rel_id,edl_valid_from_dts,edl_valid_to_dts,ikb_no,del_f
0,11410074,2015-04-02 16:38:24,2016-01-25 18:14:36,1-7JI-23579,N
1,11410074,2016-01-25 18:14:36,2016-02-14 05:19:56.921842,1-7JI-23579,N
2,11410074,2016-02-14 05:19:56.921842,2016-03-27 08:45:36,1-7JI-23579,N
3,11410074,2016-03-27 08:45:36,2016-04-01 12:59:00,1-7JI-23579,N
4,11410074,2016-04-01 12:59:00,2016-09-24 06:35:53.613126,1-7JI-23579,N
...,...,...,...,...,...
110,118492640,2024-04-24 16:33:50,2024-11-28 05:48:49.831784,1-1TWAZ5GR,N
111,118492640,2024-11-28 05:48:49.831784,2025-02-19 11:10:02.333993,1-1TWAZ5GR,N
112,118492640,2025-02-19 11:10:02.333993,2025-10-01 07:27:12.034435,1-1TWAZ5GR,N
113,118492640,2025-10-01 07:27:12.034435,9999-12-31 00:00:00,1-1TWAZ5GR,N


output


,REL_ID_REGIE_KLANT,REL_ID
0,118220032,105181982
1,118220032,115379534
2,118220032,118220032
3,130000001,130000001
4,118220032,130000003


Case 6: PASS
Case 6: output has no duplicate rows


## Case 7: new rel_id for an existing ikb_no

Scenario:
- insert a new `ggm_np.rel_id` that reuses existing `ikb_no='1-1N09820A'`

Expected delta/output:
- inserted: `('130000004', '118220032')`
- removed: none

In [47]:
prev_pairs = pair_set(spark.table(f"{WORK_DB}.output"))

spark.sql(
    f"""
    INSERT INTO {WORK_DB}.ggm_np (rel_id, edl_valid_from_dts, edl_valid_to_dts, ikb_no, del_f)
    VALUES (130000004, to_timestamp('2026-05-18 12:30:00'), to_timestamp('9999-12-31 00:00:00'), '1-1N09820A', 'N')
    """
)

_ = run_once()
cur_pairs = pair_set(spark.table(f"{WORK_DB}.output"))
show_case_tables('Case 7')
assert_delta('Case 7', prev_pairs, cur_pairs, [('130000004', '118220032')], [])
assert_output_has_no_duplicates('Case 7')

Case 7 input/output preview
direct_bank


,rel_id,edl_valid_from_dts,edl_valid_to_dts,np_sbl_id,drc_bnk_f
0,11410074,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-7JI-23579,Y
1,11410074,2020-03-05 10:51:06,2023-08-28 07:05:17,1-7JI-23579,N
2,11410074,2023-08-28 07:05:17,9999-12-31 00:00:00,1-7JI-23579,N
3,117786942,2019-11-08 18:15:22.600224,2024-04-09 11:00:16.23149,1-1B89RM5A,N
4,117786942,2024-04-09 11:00:16.23149,9999-12-31 00:00:00,1-1B89RM5A,N
5,118220032,2019-11-08 18:15:22.600224,9999-12-31 00:00:00,1-1N09820A,Y
6,118492640,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-1TWAZ5GR,N
7,118492640,2020-03-05 10:51:06,2024-04-09 11:00:15,1-1TWAZ5GR,Y
8,118492640,2024-04-09 11:00:15,9999-12-31 00:00:00,1-1TWAZ5GR,N
9,130000001,2026-05-18 09:00:00,9999-12-31 00:00:00,1-Z-NEW1,Y


ggm_np


,rel_id,edl_valid_from_dts,edl_valid_to_dts,ikb_no,del_f
0,11410074,2015-04-02 16:38:24,2016-01-25 18:14:36,1-7JI-23579,N
1,11410074,2016-01-25 18:14:36,2016-02-14 05:19:56.921842,1-7JI-23579,N
2,11410074,2016-02-14 05:19:56.921842,2016-03-27 08:45:36,1-7JI-23579,N
3,11410074,2016-03-27 08:45:36,2016-04-01 12:59:00,1-7JI-23579,N
4,11410074,2016-04-01 12:59:00,2016-09-24 06:35:53.613126,1-7JI-23579,N
...,...,...,...,...,...
111,118492640,2024-11-28 05:48:49.831784,2025-02-19 11:10:02.333993,1-1TWAZ5GR,N
112,118492640,2025-02-19 11:10:02.333993,2025-10-01 07:27:12.034435,1-1TWAZ5GR,N
113,118492640,2025-10-01 07:27:12.034435,9999-12-31 00:00:00,1-1TWAZ5GR,N
114,130000003,2026-05-18 11:00:00,9999-12-31 00:00:00,1-1N09820A,N


output


,REL_ID_REGIE_KLANT,REL_ID
0,118220032,105181982
1,118220032,115379534
2,118220032,118220032
3,130000001,130000001
4,118220032,130000003
5,118220032,130000004


Case 7: PASS
Case 7: output has no duplicate rows


## Case 8: simultaneous new relationships in both tables

Scenario:
- insert a new active `direct_bank` seed and a new `ggm_np` row at the same time
- both rows share the same business identifier, so the new `ggm_np` relation should inherit the new seed

Expected delta/output:
- inserted: `('130000010', '130000010')`, `('130000011', '130000010')`
- removed: none

In [48]:
prev_pairs = pair_set(spark.table(f"{WORK_DB}.output"))

spark.sql(
    f"""
    INSERT INTO {WORK_DB}.direct_bank (rel_id, edl_valid_from_dts, edl_valid_to_dts, np_sbl_id, drc_bnk_f)
    VALUES (130000010, to_timestamp('2026-05-18 13:00:00'), to_timestamp('9999-12-31 00:00:00'), '1-CROSS-LINK-01', 'Y')
    """
)
spark.sql(
    f"""
    INSERT INTO {WORK_DB}.ggm_np (rel_id, edl_valid_from_dts, edl_valid_to_dts, ikb_no, del_f)
    VALUES (130000011, to_timestamp('2026-05-18 13:05:00'), to_timestamp('9999-12-31 00:00:00'), '1-CROSS-LINK-01', 'N')
    """
)

_ = run_once()
cur_pairs = pair_set(spark.table(f"{WORK_DB}.output"))
show_case_tables('Case 8')
assert_delta('Case 8', prev_pairs, cur_pairs, [('130000010', '130000010'), ('130000011', '130000010')], [])
assert_output_has_no_duplicates('Case 8')

Case 8 input/output preview
direct_bank


,rel_id,edl_valid_from_dts,edl_valid_to_dts,np_sbl_id,drc_bnk_f
0,11410074,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-7JI-23579,Y
1,11410074,2020-03-05 10:51:06,2023-08-28 07:05:17,1-7JI-23579,N
2,11410074,2023-08-28 07:05:17,9999-12-31 00:00:00,1-7JI-23579,N
3,117786942,2019-11-08 18:15:22.600224,2024-04-09 11:00:16.23149,1-1B89RM5A,N
4,117786942,2024-04-09 11:00:16.23149,9999-12-31 00:00:00,1-1B89RM5A,N
5,118220032,2019-11-08 18:15:22.600224,9999-12-31 00:00:00,1-1N09820A,Y
6,118492640,2019-11-08 18:15:22.600224,2020-03-05 10:51:06,1-1TWAZ5GR,N
7,118492640,2020-03-05 10:51:06,2024-04-09 11:00:15,1-1TWAZ5GR,Y
8,118492640,2024-04-09 11:00:15,9999-12-31 00:00:00,1-1TWAZ5GR,N
9,130000001,2026-05-18 09:00:00,9999-12-31 00:00:00,1-Z-NEW1,Y


ggm_np


,rel_id,edl_valid_from_dts,edl_valid_to_dts,ikb_no,del_f
0,11410074,2015-04-02 16:38:24,2016-01-25 18:14:36,1-7JI-23579,N
1,11410074,2016-01-25 18:14:36,2016-02-14 05:19:56.921842,1-7JI-23579,N
2,11410074,2016-02-14 05:19:56.921842,2016-03-27 08:45:36,1-7JI-23579,N
3,11410074,2016-03-27 08:45:36,2016-04-01 12:59:00,1-7JI-23579,N
4,11410074,2016-04-01 12:59:00,2016-09-24 06:35:53.613126,1-7JI-23579,N
...,...,...,...,...,...
112,118492640,2025-02-19 11:10:02.333993,2025-10-01 07:27:12.034435,1-1TWAZ5GR,N
113,118492640,2025-10-01 07:27:12.034435,9999-12-31 00:00:00,1-1TWAZ5GR,N
114,130000003,2026-05-18 11:00:00,9999-12-31 00:00:00,1-1N09820A,N
115,130000004,2026-05-18 12:30:00,9999-12-31 00:00:00,1-1N09820A,N


output


,REL_ID_REGIE_KLANT,REL_ID
0,118220032,105181982
1,118220032,115379534
2,118220032,118220032
3,130000001,130000001
4,118220032,130000003
5,118220032,130000004
6,130000010,130000010
7,130000010,130000011


Case 8: PASS
Case 8: output has no duplicate rows
